# PyRosetta Scoring Functions

This notebook demonstrates the three PyRosetta scoring operations wrapped in `proto_tools`: Spatial Aggregation Propensity (SAP), Solvent Accessible Surface Area (SASA), and full Rosetta energy scoring. Each tool takes one or more protein structures and returns physics-based numeric scores with per-residue breakdowns where applicable.

**License:** PyRosetta requires acceptance of the [Rosetta Software License](https://www.rosettacommons.org/software/license-and-download). Free for academic and non-commercial use; commercial users must obtain a separate license.

In [56]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("pyrosetta")
display_overview("pyrosetta")
display_docs_section("pyrosetta", "Background")

# PyRosetta Scoring Functions

> PyRosetta provides physics-based scoring of protein structures using the Rosetta molecular modeling suite. Three operations are available: Spatial Aggregation Propensity (SAP) scoring, Solvent Accessible Surface Area (SASA) computation, and full Rosetta energy scoring with optional FastRelax.

## Background

**Spatial Aggregation Propensity (SAP)** quantifies how much hydrophobic surface area is exposed on a protein. Proteins with large patches of exposed hydrophobicity are prone to [aggregation](https://en.wikipedia.org/wiki/Protein_aggregation) -- a major concern in therapeutic protein development. SAP computes a per-atom hydrophobicity score weighted by solvent exposure, then sums across the surface. Higher SAP scores indicate greater aggregation risk.

**Solvent Accessible Surface Area (SASA)** measures the total surface area of a protein that is accessible to solvent molecules (modeled as a spherical probe, typically 1.4 A radius for water). SASA is fundamental to understanding protein folding thermodynamics: buried residues contribute to the [hydrophobic core](https://en.wikipedia.org/wiki/Hydrophobic_core), while exposed residues interact with solvent and binding partners. Per-residue SASA values reveal which positions are buried vs. solvent-exposed.

**Rosetta Energy Scoring** evaluates protein structures using a physics-based [energy function](https://en.wikipedia.org/wiki/Force_field_\(chemistry\)) that combines van der Waals interactions, electrostatics, hydrogen bonding, solvation, and backbone torsion preferences. The score is reported in Rosetta Energy Units (REU). Lower (more negative) total energies indicate more favorable structures. [FastRelax](https://www.rosettacommons.org/docs/latest/scripting_documentation/RosettaScripts/Movers/movers_pages/FastRelaxMover) is optionally applied before scoring to resolve steric clashes and backbone strain.

## Available tools

In [57]:
display_available_tools("pyrosetta")

- **`run_pyrosetta_energy()`** — Compute Rosetta energy scores for protein structures with optional FastRelax
- **`run_pyrosetta_sap()`** — Compute Spatial Aggregation Propensity (SAP) scores for protein structures using PyRosetta
- **`run_pyrosetta_sasa()`** — Compute Solvent Accessible Surface Area (SASA) for protein structures using PyRosetta

### `run_pyrosetta_relax`

FastRelax minimizes a structure under the chosen score function and returns the relaxed coordinates as a `Structure`. Use this when downstream filters or scorers need to operate on a relaxed pose (e.g. cofolding filter pipelines for binder design). The relaxed `Structure` chains directly into the other PyRosetta scorers or into geometric `Structure` methods.


In [58]:
from proto_tools import (
    PyRosettaEnergyConfig,
    PyRosettaEnergyInput,
    PyRosettaRelaxConfig,
    PyRosettaRelaxInput,
    run_pyrosetta_energy,
    run_pyrosetta_relax,
)
from proto_tools.tools.structure_scoring.pyrosetta.pyrosetta_relax import example_input as relax_example_input


In [59]:
# Display input docs
display_api_reference("pyrosetta", "input", "run_pyrosetta_relax")

relax_inputs = relax_example_input()


**Input** — `PyRosettaEnergyInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `inputs` | `List[ScoringStructureInput]` | required | Protein structures to score, each with optional chain selection. Accepts bare Structure objects, PDB file paths, or PDB content strings for convenience. |
| `structure` | `Structure` | required | The protein structure. Accepts file path, PDB content string, or Structure object. |
| `chain_ids` | `array` |  | Optional list of chain IDs to include in scoring. If None, all chains are included. |

In [60]:
# Display config docs
display_api_reference("pyrosetta", "config", "run_pyrosetta_relax")

# Single FastRelax cycle to match Germinal's `-relax:default_repeats 1` and keep the example fast
relax_config = PyRosettaRelaxConfig(scorefxn="ref2015", relax_cycles=1)


**Config** — `PyRosettaEnergyConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `scorefxn` | `string` | `ref2015` | Rosetta score function name. 'ref2015' is the current community standard. |
| `relax` | `boolean` | `True` | Whether to run FastRelax before scoring. Recommended for raw PDB structures to resolve clashes and strain. |
| `relax_cycles` | `integer` | `5` | Number of FastRelax cycles. More cycles improve convergence but increase runtime. |
| `constrain_to_start` | `boolean` | `True` | Whether to constrain relaxation to starting coordinates. Prevents large structural deviations. |
| `seed` | `integer` |  | Random seed. When set, tools run reproducibly up to small GPU float noise (see `BaseToolOutput.approx_equal`). When None, tools take their fast non-deterministic path. |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cpu` | Device to run the tool on. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [61]:
# Run FastRelax
relax_result = run_pyrosetta_relax(relax_inputs, relax_config)


Running run_pyrosetta_relax [00:00]

In [62]:
# Display output docs and inspect results
display_api_reference("pyrosetta", "output", "run_pyrosetta_relax")

r = relax_result.results[0]
print(f"Total score (relaxed): {r.total_score:.1f} REU")
print(f"Cycles applied:        {r.relax.relax_cycles}")
print(f"Output chains:         {r.relax.relaxed_structure.get_chain_ids()}")

# The relaxed Structure is ready to chain into another scorer.
relaxed_structure = r.relax.relaxed_structure
relaxed_structure.visualize(color_by="chain")


**Output** — `PyRosettaEnergyOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `results` | `List[PyRosettaEnergyMetrics]` |  | Energy scores, one per input structure. Each entry carries `total_energy` + `relaxed` as specced metrics plus `energy_terms` and `per_residue` as declared non-metric fields. |
| `energy_terms` | `Dict[string, number]` | required | Breakdown by score term (fa\_atr, fa\_rep, etc.). Always the whole-pose terms. Declared as a real field (not a metric) because it's a named-term breakdown, not a scalar quantity. |
| `per_residue` | `List[ResidueEnergy]` | required | Per-residue energy breakdown, filtered to the selected chains when `chain_ids` is set. Declared as a real field because each entry carries chain/residue identifiers alongside the energy value. |
| `primary_metric` | `string` |  | Name of the metric that best summarizes the result overall (e.g. `"avg_plddt"` for AlphaFold2). Used by downstream UI and reporting to pick a headline value. ### `run_pyrosetta_sap()` Compute Spatial Aggregation Propensity (SAP) scores for protein structures using PyRosetta |

Total score (relaxed): -187.2 REU
Cycles applied:        1
Output chains:         ['A']


In [63]:
# Chain the relaxed Structure into pyrosetta-energy. Default config = no further relax.
chained = run_pyrosetta_energy(
    PyRosettaEnergyInput(inputs=[relaxed_structure]),
)
print(f"Re-scored relaxed pose: {chained.results[0].total_energy:.1f} REU")


Running run_pyrosetta_energy [00:00]

Re-scored relaxed pose: -187.2 REU


### `run_pyrosetta_sap`

SAP (Spatial Aggregation Propensity) quantifies how much hydrophobic surface area is exposed on a protein. Proteins with large patches of exposed hydrophobicity are prone to aggregation — a major concern in therapeutic protein development. `run_pyrosetta_sap` loads each input structure into a Rosetta Pose and scores it with Rosetta's `core.pack.guidance_scoreterms.sap` module, returning a single scalar SAP score per structure. Higher SAP scores indicate greater aggregation risk.

In [64]:
from proto_tools import (
    PyRosettaSAPConfig,
    PyRosettaSAPInput,
    run_pyrosetta_sap,
)
from proto_tools.tools.structure_scoring.pyrosetta.pyrosetta_sap import example_input as sap_example_input

In [65]:
# Display input docs
display_api_reference("pyrosetta", "input", "run_pyrosetta_sap")

# Use the tool's canonical example input PDB.
sap_inputs = sap_example_input()

# Let's take a look at what this looks like
sap_inputs.inputs[0].structure.visualize(color_by="chain")

**Input** — `PyRosettaSAPInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `inputs` | `List[ScoringStructureInput]` | required | Protein structures to score, each with optional chain selection. Accepts bare Structure objects, PDB file paths, or PDB content strings for convenience. |
| `structure` | `Structure` | required | The protein structure. Accepts file path, PDB content string, or Structure object. |
| `chain_ids` | `array` |  | Optional list of chain IDs to include in scoring. If None, all chains are included. |

In [66]:
# Display config docs
display_api_reference("pyrosetta", "config", "run_pyrosetta_sap")

# PyRosettaSAPConfig has no tool-specific parameters; all defaults are used
sap_config = PyRosettaSAPConfig()

**Config** — `PyRosettaSAPConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `seed` | `integer` |  | Random seed. When set, tools run reproducibly up to small GPU float noise (see `BaseToolOutput.approx_equal`). When None, tools take their fast non-deterministic path. |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cpu` | Device to run the tool on. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [67]:
# Run the SAP scoring function
sap_result = run_pyrosetta_sap(sap_inputs, sap_config)

Running run_pyrosetta_sap [00:00]

In [68]:
# Display output docs and inspect results
display_api_reference("pyrosetta", "output", "run_pyrosetta_sap")

for r in sap_result.results:
    print(f"SAP score: {r.sap_score:.2f}")

**Output** — `PyRosettaSAPOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `results` | `List[PyRosettaSAPMetrics]` |  | SAP scores with per-residue breakdown, one per input structure. |
| `per_residue` | `List[ResidueSAP]` | required | Per-residue SAP contributions. Declared as a real field (not a metric) because each entry carries chain/residue identifiers alongside the score. |
| `primary_metric` | `string` |  | Name of the metric that best summarizes the result overall (e.g. `"avg_plddt"` for AlphaFold2). Used by downstream UI and reporting to pick a headline value. |

SAP score: 50.62


### `run_pyrosetta_sasa`

SASA (Solvent Accessible Surface Area) measures the total protein surface accessible to a spherical solvent probe (1.4 A by default, matching water). Buried residues contribute to the hydrophobic core, while exposed residues interact with solvent and binding partners. `run_pyrosetta_sasa` returns both the total SASA and a per-residue breakdown, which is useful for identifying surface-exposed sites, characterizing the hydrophobic core, or feeding into downstream feature engineering.

In [69]:
from proto_tools import (
    PyRosettaSASAConfig,
    PyRosettaSASAInput,
    run_pyrosetta_sasa,
)
from proto_tools.tools.structure_scoring.pyrosetta.pyrosetta_sasa import example_input as sasa_example_input

In [70]:
# Display input docs
display_api_reference("pyrosetta", "input", "run_pyrosetta_sasa")

sasa_inputs = sasa_example_input()

**Input** — `PyRosettaSASAInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `inputs` | `List[ScoringStructureInput]` | required | Protein structures to analyze, each with optional chain selection. Accepts bare Structure objects, PDB file paths, or PDB content strings for convenience. |
| `structure` | `Structure` | required | The protein structure. Accepts file path, PDB content string, or Structure object. |
| `chain_ids` | `array` |  | Optional list of chain IDs to include in scoring. If None, all chains are included. |

In [71]:
# Display config docs
display_api_reference("pyrosetta", "config", "run_pyrosetta_sasa")

# 1.4 A is the standard water probe radius
sasa_config = PyRosettaSASAConfig(probe_radius=1.4)

**Config** — `PyRosettaSASAConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `probe_radius` | `number` | `1.4` | Radius of the solvent probe sphere in Angstroms. Standard water probe is 1.4 A. |
| `seed` | `integer` |  | Random seed. When set, tools run reproducibly up to small GPU float noise (see `BaseToolOutput.approx_equal`). When None, tools take their fast non-deterministic path. |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cpu` | Device to run the tool on. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [72]:
# Run the SASA computation
sasa_result = run_pyrosetta_sasa(sasa_inputs, sasa_config)

Running run_pyrosetta_sasa [00:00]

In [73]:
# Display output docs
display_api_reference("pyrosetta", "output", "run_pyrosetta_sasa")

# Total SASA plus the five most solvent-exposed residues
r = sasa_result.results[0]
print(f"Total SASA: {r.total_sasa:.1f} A^2")
print(f"Residues:   {len(r.per_residue)}")

top_exposed = sorted(r.per_residue, key=lambda res: res.sasa, reverse=True)[:5]
print("\nTop 5 most exposed residues:")
for res in top_exposed:
    print(f"  {res.chain_id}:{res.residue_name}{res.residue_index} — {res.sasa:.1f} A^2")

**Output** — `PyRosettaSASAOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `results` | `List[PyRosettaSASAMetrics]` |  | SASA results, one per input structure. |
| `per_residue` | `List[ResidueSASA]` | required | Per-residue SASA breakdown. Declared as a real field (not a metric) because each entry carries chain/residue identifiers alongside the SASA value. |
| `primary_metric` | `string` |  | Name of the metric that best summarizes the result overall (e.g. `"avg_plddt"` for AlphaFold2). Used by downstream UI and reporting to pick a headline value. |

Total SASA: 4978.9 A^2
Residues:   65

Top 5 most exposed residues:
  A:LYS46 — 235.9 A^2
  A:ARG21 — 214.8 A^2
  A:MET1 — 195.9 A^2
  A:PRO65 — 186.1 A^2
  A:LYS35 — 179.9 A^2


### `run_pyrosetta_energy`

Rosetta energy scoring evaluates a protein structure with a physics-based energy function that combines van der Waals interactions, electrostatics, hydrogen bonding, solvation, and backbone torsion terms. The score is reported in Rosetta Energy Units (REU); lower (more negative) is more favorable. By default, `run_pyrosetta_energy` scores the input structure as-given; setting `pre_relax_structures=True` on the config opts into the [FastRelax](https://www.rosettacommons.org/docs/latest/scripting_documentation/RosettaScripts/Movers/movers_pages/FastRelaxMover) preprocess (composing `pyrosetta-relax` under the hood) to resolve clashes and backbone strain first. Returns the total energy, a per-term breakdown, and per-residue totals. The default score function is `ref2015`, Rosetta's current production all-atom energy function.


In [74]:
from proto_tools import (
    PyRosettaEnergyConfig,
    PyRosettaEnergyInput,
    PyRosettaRelaxConfig,
    run_pyrosetta_energy,
)
from proto_tools.tools.structure_scoring.pyrosetta.pyrosetta_energy import example_input as energy_example_input


In [75]:
# Display input docs
display_api_reference("pyrosetta", "input", "run_pyrosetta_energy")

energy_inputs = energy_example_input()

**Input** — `PyRosettaEnergyInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `inputs` | `List[ScoringStructureInput]` | required | Protein structures to score, each with optional chain selection. Accepts bare Structure objects, PDB file paths, or PDB content strings for convenience. |
| `structure` | `Structure` | required | The protein structure. Accepts file path, PDB content string, or Structure object. |
| `chain_ids` | `array` |  | Optional list of chain IDs to include in scoring. If None, all chains are included. |

In [76]:
# Display config docs
display_api_reference("pyrosetta", "config", "run_pyrosetta_energy")

# Use ref2015 with the FastRelax preprocess (1 cycle to keep the example fast)
energy_config = PyRosettaEnergyConfig(
    scorefxn="ref2015",
    pre_relax_structures=True,
    relax_config=PyRosettaRelaxConfig(relax_cycles=1),
)


**Config** — `PyRosettaEnergyConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `scorefxn` | `string` | `ref2015` | Rosetta score function name. 'ref2015' is the current community standard. |
| `relax` | `boolean` | `True` | Whether to run FastRelax before scoring. Recommended for raw PDB structures to resolve clashes and strain. |
| `relax_cycles` | `integer` | `5` | Number of FastRelax cycles. More cycles improve convergence but increase runtime. |
| `constrain_to_start` | `boolean` | `True` | Whether to constrain relaxation to starting coordinates. Prevents large structural deviations. |
| `seed` | `integer` |  | Random seed. When set, tools run reproducibly up to small GPU float noise (see `BaseToolOutput.approx_equal`). When None, tools take their fast non-deterministic path. |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cpu` | Device to run the tool on. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [77]:
# Run the energy scoring function
energy_result = run_pyrosetta_energy(energy_inputs, energy_config)

Running run_pyrosetta_relax [00:00]

Running run_pyrosetta_energy [00:00]

In [78]:
# Display output docs
display_api_reference("pyrosetta", "output", "run_pyrosetta_energy")

r = energy_result.results[0]
print(f"Total energy: {r.total_energy:.1f} REU")
print("\nTop contributing energy terms:")
for term, value in sorted(r.energy_terms.items(), key=lambda kv: kv[1])[:6]:
    print(f"  {term:>12s}: {value:.2f}")


**Output** — `PyRosettaEnergyOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `results` | `List[PyRosettaEnergyMetrics]` |  | Energy scores, one per input structure. Each entry carries `total_energy` + `relaxed` as specced metrics plus `energy_terms` and `per_residue` as declared non-metric fields. |
| `energy_terms` | `Dict[string, number]` | required | Breakdown by score term (fa\_atr, fa\_rep, etc.). Always the whole-pose terms. Declared as a real field (not a metric) because it's a named-term breakdown, not a scalar quantity. |
| `per_residue` | `List[ResidueEnergy]` | required | Per-residue energy breakdown, filtered to the selected chains when `chain_ids` is set. Declared as a real field because each entry carries chain/residue identifiers alongside the energy value. |
| `primary_metric` | `string` |  | Name of the metric that best summarizes the result overall (e.g. `"avg_plddt"` for AlphaFold2). Used by downstream UI and reporting to pick a headline value. |

Total energy: -189.1 REU

Top contributing energy terms:
        fa_atr: -337.64
       fa_elec: -115.97
       p_aa_pp: -30.98
   hbond_sr_bb: -21.73
   hbond_lr_bb: -15.62
      hbond_sc: -8.39


## Export Results

Each output model supports export to common file formats for downstream analysis. SASA results export well to CSV because of the per-residue table; energy results export to JSON to preserve the nested per-term breakdown.

In [79]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

sap_result.export("pyrosetta_sap", export_path=output_dir, file_format="json")
sasa_result.export("pyrosetta_sasa", export_path=output_dir, file_format="csv")
energy_result.export("pyrosetta_energy", export_path=output_dir, file_format="json")

for name in ("pyrosetta_sap.json", "pyrosetta_sasa.csv", "pyrosetta_energy.json"):
    print(f"Exported {output_dir / name}")

Exported example_output/pyrosetta_sap.json
Exported example_output/pyrosetta_sasa.csv
Exported example_output/pyrosetta_energy.json
